# ShelfWise - Full-application stress test under the world simulation

**Run every cell top to bottom (Kernel -> Restart & Run All).** This does not test the fine-tuned
model in isolation - it drives the *whole running application* the way it would actually be used:
the world simulation (`src/shelfwise_worldgen/`) generates fresh synthetic retail events across a
realistic, full-supermarket product assortment (thousands of SKUs spanning dozens of departments -
produce, dairy, electronics, pharmacy, alcohol, everything - not just the one hero SKU), every event
goes through the real ingest -> cascade -> Critic -> Executive -> HITL pipeline, and periodically a
natural-language question about whatever the simulation just generated is sent through the real chat
endpoint, so the app is being exercised end to end, not just the model.

**Important correction from earlier guidance:** the deterministic decision-science cascade (the part
the world simulation drives) never calls any LLM at all, by design - the math is meant to be
auditable and testable on its own, not guessed by a model. The **only** place this app actually calls
an LLM is the chat feature, and it always calls the "strong" model tier. That means the environment
variable that actually matters for exercising your fine-tuned model here is **`LLM_STRONG_MODEL`**,
not `LLM_ROUTINE_MODEL` as suggested in the fine-tune notebook's final instructions.

**Before running, in a terminal in this same pod:**
1. Make sure `vllm serve ... --enable-lora --lora-modules shelfwise=<adapter_dir> ...` (printed at
   the end of `02_shelfwise_gemma_finetune.ipynb`) is already running.
2. Set these two environment variables **before starting this notebook's kernel**:
   - `LLM_BASE_URL=http://127.0.0.1:8000`
   - `LLM_STRONG_MODEL=shelfwise`

If you skip that, this notebook still runs correctly and still stress-tests the whole application -
chat answers just come from the deterministic offline fallback instead of your fine-tuned model. This
notebook checks for and reports that explicitly, it does not fail silently.

Set these to control scope:

- `STRESS_HOURS` (default `3.0`) - wall-clock budget for the whole stress loop.
- `STRESS_ASSORTMENT_SIZE` (default `3000`) - how many products (out of the full generated catalog)
  make up the simulated store's assortment each cycle.
- `STRESS_CATALOG_SCALE` (default `supermarket`) - `convenience` (~1,035 possible products),
  `supermarket` (~7,254), or `hypermarket` (~24,675) - the pool `STRESS_ASSORTMENT_SIZE` draws from.
- `STRESS_CHAT_EVERY_N_CYCLES` (default `3`) - how often (in world-sim cycles) to also fire a sample
  chat question about a product from that cycle's assortment.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

STRESS_HOURS = float(os.environ.get("STRESS_HOURS", "3.0"))
STRESS_ASSORTMENT_SIZE = int(os.environ.get("STRESS_ASSORTMENT_SIZE", "3000"))
STRESS_CATALOG_SCALE = os.environ.get("STRESS_CATALOG_SCALE", "supermarket")
STRESS_CHAT_EVERY_N_CYCLES = int(os.environ.get("STRESS_CHAT_EVERY_N_CYCLES", "3"))
BASE_SEED = 20260709

print(f"repo root: {ROOT}")
print(f"stress hours: {STRESS_HOURS}")
print(f"assortment size: {STRESS_ASSORTMENT_SIZE} (scale: {STRESS_CATALOG_SCALE})")
print(f"chat sample every {STRESS_CHAT_EVERY_N_CYCLES} cycles")

RESULTS: list[dict] = []


def record(name: str, ok: bool, detail: str = "") -> None:
    RESULTS.append({"section": name, "ok": ok, "detail": detail})
    banner = "PASS" if ok else "FAIL"
    print(f"\n[{banner}] {name}" + (f" - {detail}" if detail else ""))


def run(name: str, fn) -> None:
    try:
        fn()
        record(name, True)
    except AssertionError as exc:
        record(name, False, f"assertion failed: {exc}")
    except Exception as exc:  # noqa: BLE001 - deliberately broad, this is a test harness
        record(name, False, f"{type(exc).__name__}: {exc}")


## 1. Build the full-supermarket assortment (report coverage up front)

Uses `shelfwise_worldgen.catalog.sample.sample_assortment` directly - the same deterministic
generator the API uses internally - purely to report *how broad* this run's coverage actually is
before spending any time on it: how many departments, how many distinct product names, real examples
from outside the usual dairy/hero-SKU corner of the catalog.

In [ ]:
from shelfwise_worldgen.catalog.sample import sample_assortment


def _assortment_report() -> None:
    assortment = sample_assortment(BASE_SEED, size=STRESS_ASSORTMENT_SIZE, scale=STRESS_CATALOG_SCALE)
    departments = sorted({product.department for product in assortment})
    categories = sorted({product.category for product in assortment})

    print(f"assortment size: {len(assortment)}")
    print(f"departments covered: {len(departments)}")
    print(f"categories covered: {len(categories)}")
    print("sample departments:", departments[:15])

    non_food_hit = any(
        department in {"Electronics", "Pharmacy", "Alcohol", "Clothing", "Hardware"}
        for department in departments
    )
    print("touches non-food departments (electronics/pharmacy/alcohol/etc):", non_food_hit)

    assert len(assortment) == STRESS_ASSORTMENT_SIZE
    assert len(departments) > 10, "expected broad department coverage, not just a few categories"


run("Full-supermarket assortment coverage report", _assortment_report)


## 2. Check whether the fine-tuned model is actually wired in (informational, non-fatal)

This does not fail the notebook either way - it just tells you plainly which one you are about to
stress-test: your fine-tuned local model, or the deterministic offline fallback. If this says
"offline", the loop below still runs and still stress-tests the whole application - the chat answers
just will not be your model's.

In [ ]:
from shelfwise_inference import load_inference_config


def _chat_model_wiring_check() -> None:
    config = load_inference_config()
    print(json.dumps(config.to_public_dict(), indent=2))
    if config.provider.value == "offline":
        print(
            "\nchat will use the deterministic OFFLINE fallback, not a real model - "
            "set LLM_BASE_URL and LLM_STRONG_MODEL before restarting the kernel if you "
            "want this run to exercise your fine-tuned model."
        )
    else:
        print(f"\nchat will call: provider={config.provider.value} strong_model={config.strong_model}")


run("Chat model wiring check", _chat_model_wiring_check)


## 3. The stress loop - autonomous, no human in the loop

Each cycle rotates through every registered scenario (payday cold-chain crisis, mid-month lull, stage-6 blackout weekend), a rotation of assortment sizes, and catalog scales - so a long run sweeps genuinely different store conditions instead of repeating one story. Every event goes through the real ingest -> cascade -> Critic pipeline, and then the **autopilot reviewer** (`shelfwise_eval.autopilot`) resolves every pending decision through the same public HITL endpoints a human would use - a fixed, documented policy: approve what the critic approved, approve small-exposure reviews, reject large-exposure reviews and anything the critic rejected. No decision is left hanging, and the learning loop actually fires.

**Every decision leaves a trail**: each one is appended to `data/harness_runs/<run_id>/stress_test/decision_trail.jsonl` *as it happens* (crash-safe - an interruption in hour 7 keeps hours 1-6), with the cycle, scenario, autopilot verdict and reason, and the resolved status.

In [ ]:
import os

# Unattended soak runs push write rates far past interactive defaults; these knobs exist
# for exactly this. Must be set BEFORE the first shelfwise_backend.app import in this kernel.
os.environ.setdefault("SHELFWISE_WRITE_RATE_CAPACITY", "1000000")
os.environ.setdefault("SHELFWISE_WRITE_RATE_REFILL_PER_S", "50000")

from fastapi.testclient import TestClient
from shelfwise_backend.app import app
from shelfwise_eval.autopilot import APPROVE, REJECT, review_decision
from shelfwise_worldgen.scenarios import SCENARIOS

OFFLINE_MARKERS = ("Current ShelfWise state:", "The current recommendation is", "ShelfWise is tracking")


def _looks_offline(answer: str) -> bool:
    return any(answer.startswith(marker) for marker in OFFLINE_MARKERS)


def _sample_chat_question(product) -> str:
    templates = (
        f"Why might {product.name} in {product.department} need attention right now?",
        f"What is the current risk profile for {product.name} ({product.subcategory})?",
        f"Should we reorder or markdown {product.name} today, and why?",
    )
    return templates[hash(product.product_id) % len(templates)]


def _resolve_with_autopilot(client, decision, totals, trail_lines, context) -> None:
    verdict = review_decision(decision)
    resolved_status = decision.get("status")
    if verdict["action"] in (APPROVE, REJECT):
        response = client.post(f"/decisions/{decision['id']}/{verdict['action']}")
        if response.status_code == 429:
            time.sleep(0.5)
            response = client.post(f"/decisions/{decision['id']}/{verdict['action']}")
        if response.status_code == 200:
            resolved_status = response.json()["decision"]["status"]
            totals["approved" if verdict["action"] == APPROVE else "rejected"] += 1
            expected_terminal = "approved" if verdict["action"] == APPROVE else "rejected"
            if resolved_status != expected_terminal:
                totals["state_mismatches"] += 1
        else:
            totals["autopilot_errors"] += 1
    action = decision.get("action") if isinstance(decision.get("action"), dict) else {}
    action_type = str(action.get("type") or "unknown")
    action_counts[action_type] = action_counts.get(action_type, 0) + 1
    trail_lines.append(
        {
            **context,
            "decision_id": decision.get("id"),
            "role": decision.get("role"),
            "action_type": action.get("type"),
            "critic_verdict": decision.get("critic_verdict"),
            "autopilot": verdict,
            "resolved_status": resolved_status,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    )


def _stress_loop() -> None:
    client = TestClient(app)
    scenario_ids = list(SCENARIOS)
    sizes = [int(s) for s in os.environ.get("STRESS_ASSORTMENT_SIZES", "500,1500,3000").split(",")]
    scales = [s.strip() for s in os.environ.get("STRESS_CATALOG_SCALES", "supermarket,hypermarket").split(",")]
    run_id = os.environ.get("HARNESS_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    trail_dir = ROOT / "data" / "harness_runs" / run_id / "stress_test"
    trail_dir.mkdir(parents=True, exist_ok=True)
    trail_path = trail_dir / "decision_trail.jsonl"
    globals()["RUN_ID"] = run_id
    globals()["TRAIL_PATH"] = trail_path
    deadline = time.monotonic() + STRESS_HOURS * 3600

    totals = {
        "cycles_run": 0,
        "events_total": 0,
        "events_accepted": 0,
        "duplicates": 0,
        "decisions_total": 0,
        "approved": 0,
        "rejected": 0,
        "autopilot_errors": 0,
        "state_mismatches": 0,
        "chat_calls": 0,
        "chat_errors": 0,
        "chat_offline_answers": 0,
        "chat_model_answers": 0,
    }
    scenario_counts: dict[str, int] = {}
    action_counts: dict[str, int] = {}
    chat_latencies: list[float] = []
    cycles_path = trail_dir / "cycles.jsonl"
    cascade_counts: dict[str, int] = {}
    chat_samples: list[dict] = []
    cycle = 0

    while time.monotonic() < deadline:
        scenario_id = scenario_ids[cycle % len(scenario_ids)]
        size = sizes[cycle % len(sizes)]
        scale = scales[(cycle // len(sizes)) % len(scales)]
        seed = BASE_SEED + cycle
        assortment = sample_assortment(seed, size=size, scale=scale)

        response = client.get(
            f"/demo/worldgen/{scenario_id}",
            params={
                "seed_override": seed,
                "limit": 500,
                "assortment_size": size,
                "catalog_scale": scale,
            },
        )
        assert response.status_code == 200, response.text
        body = response.json()

        totals["cycles_run"] += 1
        totals["events_total"] += body["events_total"]
        totals["events_accepted"] += body["events_accepted"]
        totals["duplicates"] += body["duplicates"]
        totals["decisions_total"] += len(body["decisions"])
        scenario_counts[scenario_id] = scenario_counts.get(scenario_id, 0) + 1
        for cascade in body["cascades"]:
            name = cascade.get("scenario")
            if name:
                cascade_counts[name] = cascade_counts.get(name, 0) + 1

        trail_lines: list[dict] = []
        context = {"cycle": cycle, "scenario_id": scenario_id, "assortment_size": size, "catalog_scale": scale, "seed": seed}
        for decision in body["decisions"]:
            _resolve_with_autopilot(client, decision, totals, trail_lines, context)
        with trail_path.open("a", encoding="utf-8") as handle:
            for line in trail_lines:
                handle.write(json.dumps(line, sort_keys=True) + "\n")
        with cycles_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps({
                "cycle": cycle,
                "scenario_id": scenario_id,
                "assortment_size": size,
                "catalog_scale": scale,
                "seed": seed,
                "stream_events_total": body.get("stream_events_total"),
                "events_total": body["events_total"],
                "events_accepted": body["events_accepted"],
                "duplicates": body["duplicates"],
                "decisions": len(body["decisions"]),
            }, sort_keys=True) + "\n")

        if cycle % STRESS_CHAT_EVERY_N_CYCLES == 0:
            product = assortment[seed % len(assortment)]
            question = _sample_chat_question(product)
            started = time.monotonic()
            chat_response = client.post("/chat", json={"question": question})
            latency_ms = round((time.monotonic() - started) * 1000, 1)
            chat_latencies.append(latency_ms)
            totals["chat_calls"] += 1
            if chat_response.status_code != 200:
                totals["chat_errors"] += 1
            else:
                answer = chat_response.text
                if _looks_offline(answer):
                    totals["chat_offline_answers"] += 1
                else:
                    totals["chat_model_answers"] += 1
                if len(chat_samples) < 10:
                    chat_samples.append(
                        {
                            "department": product.department,
                            "product": product.name,
                            "question": question,
                            "answer": answer[:400],
                            "latency_ms": latency_ms,
                        }
                    )

        if totals["cycles_run"] % 10 == 0:
            elapsed_min = round((STRESS_HOURS * 3600 - (deadline - time.monotonic())) / 60, 1)
            print(
                f"  cycle {totals['cycles_run']}: elapsed={elapsed_min}min "
                f"events={totals['events_total']} decisions={totals['decisions_total']} "
                f"approved={totals['approved']} rejected={totals['rejected']} "
                f"chat={totals['chat_calls']}"
            )
        cycle += 1

    learning = client.get("/learning").json()
    totals["learning_events"] = len(learning["events"])
    totals["learning_thresholds"] = len(learning["thresholds"])

    print(json.dumps(totals, indent=2))
    print("scenario rotation:", json.dumps(scenario_counts, indent=2))
    print("cascade scenarios triggered:", json.dumps(cascade_counts, indent=2))
    print("sample chat exchanges:")
    print(json.dumps(chat_samples, indent=2))
    print(f"decision trail: {trail_path}")

    globals()["STRESS_TOTALS"] = totals
    globals()["STRESS_CASCADE_SCENARIOS"] = cascade_counts
    globals()["STRESS_SCENARIO_COUNTS"] = scenario_counts
    globals()["STRESS_CHAT_SAMPLES"] = chat_samples
    globals()["STRESS_ACTION_COUNTS"] = action_counts
    globals()["STRESS_CHAT_LATENCIES"] = chat_latencies

    print("decision action types:", json.dumps(action_counts, indent=2))

    assert totals["cycles_run"] > 0
    assert totals["events_total"] > 0
    assert totals["decisions_total"] > 0
    assert totals["approved"] > 0, "autopilot never approved anything - HITL loop untested"
    assert totals["autopilot_errors"] == 0, "autopilot hit HITL endpoint errors"
    assert totals["state_mismatches"] == 0, (
        f"{totals['state_mismatches']} decisions returned a different terminal state than "
        "requested - decision identity is broken; do NOT trust this run's HITL numbers"
    )
    assert len(action_counts) >= 3, (
        f"decision coverage collapsed to {sorted(action_counts)} - need >= 3 action types"
    )
    if action_counts:
        top_share = max(action_counts.values()) / max(sum(action_counts.values()), 1)
        assert top_share < 0.95, (
            f"one action type is {top_share:.0%} of all decisions - coverage too concentrated"
        )
    if os.environ.get("STRESS_REQUIRE_MODEL") == "1":
        assert totals["chat_model_answers"] > 0, (
            "STRESS_REQUIRE_MODEL=1 but zero model-backed chat answers - the model endpoint "
            "is not wired (check LLM_BASE_URL / LLM_API_KEY / LLM_STRONG_MODEL)"
        )
    assert trail_path.exists() and trail_path.stat().st_size > 0, "decision trail is empty"
    if totals["chat_calls"] > 0:
        assert totals["chat_errors"] == 0, "at least one chat call failed outright"


run("Stress loop (world sim + autopilot HITL + chat)", _stress_loop)


## 4. Export this run's stress-test data

Same convention as `01_shelfwise_full_test_harness.ipynb`'s Section 9 - written to
`data/harness_runs/<run_id>/stress_test/` so it survives the kernel stopping, but still has to be
copied off this box before the environment is reclaimed (`data/harness_runs/` is gitignored on
purpose - raw generated telemetry does not belong in git history).

In [ ]:
def _export_stress_results() -> None:
    run_id = globals().get("RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = ROOT / "data" / "harness_runs" / run_id / "stress_test"
    run_dir.mkdir(parents=True, exist_ok=True)

    totals = globals().get("STRESS_TOTALS", {})
    trail_path = globals().get("TRAIL_PATH")
    trail_lines = 0
    if trail_path is not None and Path(trail_path).exists():
        with Path(trail_path).open("r", encoding="utf-8") as handle:
            for line in handle:
                json.loads(line)
                trail_lines += 1

    from shelfwise_inference import load_inference_config

    latencies = sorted(globals().get("STRESS_CHAT_LATENCIES", []))

    def _pct(values: list[float], q: float) -> float | None:
        if not values:
            return None
        return values[min(int(len(values) * q), len(values) - 1)]

    manifest = {
        "trail_lines": trail_lines,
        "action_type_counts": globals().get("STRESS_ACTION_COUNTS", {}),
        "inference_config": load_inference_config().to_public_dict(),
        "chat_latency_ms": {
            "count": len(latencies),
            "p50": _pct(latencies, 0.50),
            "p95": _pct(latencies, 0.95),
            "max": latencies[-1] if latencies else None,
        },
        "run_id": run_id,
        "exported_at": datetime.now(timezone.utc).isoformat(),
        "stress_hours_configured": STRESS_HOURS,
        "assortment_sizes": os.environ.get("STRESS_ASSORTMENT_SIZES", "500,1500,3000"),
        "catalog_scales": os.environ.get("STRESS_CATALOG_SCALES", "supermarket,hypermarket"),
        "totals": totals,
        "scenario_rotation": globals().get("STRESS_SCENARIO_COUNTS", {}),
        "cascade_scenarios": globals().get("STRESS_CASCADE_SCENARIOS", {}),
    }
    (run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    (run_dir / "chat_samples.json").write_text(
        json.dumps(globals().get("STRESS_CHAT_SAMPLES", []), indent=2, sort_keys=True), encoding="utf-8"
    )

    from fastapi.testclient import TestClient
    from shelfwise_backend.app import app

    client = TestClient(app)
    learning = client.get("/learning").json()
    (run_dir / "learning_events.json").write_text(
        json.dumps(learning, indent=2, sort_keys=True), encoding="utf-8"
    )


    print(json.dumps(manifest, indent=2))
    print(f"\nwrote stress-test artifacts to: {run_dir}")
    assert (run_dir / "manifest.json").exists()
    assert trail_lines > 0, "decision trail must not be empty"


run("Export stress-test artifacts to disk", _export_stress_results)


## Final summary

In [ ]:
print("=" * 72)
print("SHELFWISE STRESS TEST SUMMARY")
print("=" * 72)
width = max((len(item["section"]) for item in RESULTS), default=10)
failures = 0
for item in RESULTS:
    banner = "PASS" if item["ok"] else "FAIL"
    if not item["ok"]:
        failures += 1
    line = f"{banner:4}  {item['section']:<{width}}"
    if item["detail"]:
        line += f"  ({item['detail']})"
    print(line)
print("=" * 72)
if failures == 0:
    print("ALL CHECKS PASSED - the whole application held up under the full-catalog stress test.")
else:
    print(f"{failures} check(s) FAILED - scroll up to the matching section for the full output.")
